# Lab 2 — Pydantic, Tools with Gradio
<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 250px; height: 150px; vertical-align: middle;">
            <img src="../assets/logo.png" width="250" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Objective</h2>
            <span style="color:#ff7800;">- Introduction to Pydantic Model <br>
- Introduction to Tools<br>
- Introduction to Gradio<br>
- Integration of tools with Gradio Chat
            </span>
        </td>
    </tr>
</table>



## How to Use This Notebook
We move from fundamentals to tools. Run cells in order and pause after each section to inspect the outputs.

1. Build a simple Pydantic model and confirm validation works.
2. Generate JSON schema for tools and inspect the structure.
3. Wire the tools into Ollama, handle tool calls, and return results.
4. Use the weather-only flow to see how tool usage can be constrained.

## 1. Introduction to Pydantic Models
Pydantic is Python’s most popular data validation library that uses type hints to define structured data models and automatically validate incoming data. It’s perfect for APIs, config files, and especially defining tool schemas for LLMs like those running locally with Ollama.
What Are Pydantic Models?
Pydantic models are classes that inherit from  BaseModel . They define data structure with type annotations, provide automatic validation, parsing, and serialization to/from JSON.

Key features:
	•	Runtime validation: Catches bad data immediately
	•	Type coercion:  "32"  →  32  automatically
	•	Nested models: Complex data structures
	•	JSON Schema generation: Perfect for LLMs and OpenAPI

In [1]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, Literal, Any
from datetime import date
from ollama import Client
from IPython.display import Markdown, display
import json

In [2]:
#define a basic pydantic model for user data
class User(BaseModel):
    model_config = ConfigDict(from_attributes=True)  # Replaces class Config
    
    id: int
    name: str
    email: str
    age: Optional[int] = None  # Optional with default
      

# Usage - automatic validation and type coercion
user = User(id=1, name="First name", email="dummy@example.com")
print(user.model_dump()) 

{'id': 1, 'name': 'First name', 'email': 'dummy@example.com', 'age': None}


In [4]:
user = User(id="1", name="First name", email="dummy@example.com")
print(user.model_dump()) 

{'id': 1, 'name': 'First name', 'email': 'dummy@example.com', 'age': None}


### Defining Tools for Llama/Ollama Interactions
LLMs like Llama need structured tool definitions with schemas. Pydantic models generate perfect JSON schemas automatically via  model_json_schema() .
Example: Weather Tool

### Complete Working Tool Integration

In [7]:
class WeatherParams(BaseModel):
    model_config = ConfigDict(
        schema_extra={
            "name": "get_weather",
            "description": "Get current weather data"
        }
    )
    location: str = Field(..., description="City name")
    unit: Literal["metric", "imperial"] = "metric"

In [8]:
class CalculatorParams(BaseModel):
    model_config = ConfigDict(
        schema_extra={
            "name": "calculator",
            "description": "Evaluate math expressions"
        }
    )
    expression: str = Field(..., description="Math like '2+3*4'")

In [15]:

# Build Ollama-compatible tool schemas with explicit names
def build_tool_schema(model: BaseModel, tool_name: str, tool_description: str):
    schema = model.model_json_schema()
    return {
        "type": "function",
        "function": {
            "name": tool_name,
            "description": tool_description,
            "parameters": schema,
        },
    }

# Mock tool functions
TOOLS = {
    "get_weather": {
        "schema": build_tool_schema(WeatherParams, "get_weather", "Get current weather data"),
        "func": lambda params: {
            "temperature": 22.5 if params.get("unit", "metric") == "metric" else 72.5,
            "location": params["location"]
        }
    },
    "calculator": {
        "schema": build_tool_schema(CalculatorParams, "calculator", "Evaluate math expressions"),
        "func": lambda params: {"result": eval(params["expression"])}
    }
}

In [16]:
# 3. Chat with tool calling
client = Client(host='http://localhost:11434')
model_name = 'qwen3:8b'

def chat_with_tools(prompt: str):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat(
        model=model_name,
        messages=messages,
        tools=[TOOLS["get_weather"]["schema"], TOOLS["calculator"]["schema"]]
    )
    
    tool_calls = response['message'].get('tool_calls') or []
    if not tool_calls:
        print(response['message'].get('content', ""))
        return
    
    messages.append(response['message'])
    for tool_call in tool_calls:
        tool_name = tool_call.get('function', {}).get('name', "")
        args = tool_call.get('function', {}).get('arguments', {})
        if isinstance(args, str):
            args = json.loads(args)
        if tool_name not in TOOLS:
            raise KeyError(f"Unknown tool name: {tool_name}")
        
        result = TOOLS[tool_name]["func"](args)
        print(f"Tool {tool_name} result: {result}")
        messages.append({
            "role": "tool",
            "name": tool_name,
            "content": json.dumps(result)
        })

    final_response = client.chat(
        model=model_name,
        messages=messages
    )
    print(final_response['message'].get('content', ""))

In [17]:
chat_with_tools("What's the weather in New York and what is 5*6?")

Tool get_weather result: {'temperature': 72.5, 'location': 'New York'}
Tool calculator result: {'result': 30}
The current weather in New York is **72.5°F**.  

And the result of **5 × 6** is **30**. 😊


In [30]:
import requests
import os
from typing import Dict, Any
from dotenv import load_dotenv

load_dotenv()
serper_api_key = os.getenv("SERPER_API_KEY")

def get_weather(city: str) -> Dict[str, Any]:
    """
    Fetch current weather for a given city using Serper API.
    
    Args:
        city: City name to get weather for
    
    Returns:
        Dictionary with weather data or error message
    """
    try:
        url = "https://google.serper.dev/search"
        payload = {
            "q": f"Current temperature of {city}"
        }
        headers = {
            'X-API-KEY': serper_api_key,
            'Content-Type': 'application/json'
        }
        
        response = requests.request("POST", url, headers=headers, json=payload)
        response.raise_for_status()
        
        return {
            "status": "success",
            "city": city,
            "data": response.text
        }
    except Exception as e:
        return {
            "status": "error",
            "city": city,
            "error": str(e)
        }


def convert_weather_to_markdown(weather_data: Dict[str, Any]) -> str:
    """
    Convert weather data to defined class.
    
    Args:
        weather_data: Dictionary containing weather information
    
    Returns:
        Markdown formatted string
    """
    if weather_data.get("status") == "error":
        return f"### ❌ Error\n\nFailed to get weather for {weather_data.get('city', 'Unknown')}: {weather_data.get('error', 'Unknown error')}"
    
    city = weather_data.get("city", "Unknown")
    data = weather_data.get("data", "No data")
    
    markdown = f"### 🌤️ Weather for {city}\n\n"
    markdown += f"**Raw Data:**\n```\n{data}\n```"
    markdown += "\n\n*Data provided by Serper API*"
    
    return markdown

In [31]:
# Define tool schemas for get_weather only

weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "ONLY call this tool if the user explicitly asks about weather, temperature, climate, or current conditions in a specific city. Do NOT call this for general greetings or other topics.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city name to get weather for"
                }
            },
            "required": ["city"]
        }
    }
}

# Map tool names to actual functions
TOOLS_WEATHER = {
    "get_weather": {
        "func": get_weather,
        "schema": weather_tool
    }
}


In [32]:
def chat_with_weather_tools_return(prompt: str):
    """
    Chat function that uses weather tools ONLY when asked about weather.
    Returns the markdown formatted response.
    """
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. You ONLY use the get_weather tool when the user explicitly asks about weather,\
                  temperature, climate, or current conditions in a specific city. For all other questions,\
                      respond normally without using any tools."
        },
        {"role": "user", "content": prompt}
    ]
    
    response = client.chat(
        model=model_name,
        messages=messages,
        tools=[weather_tool]
    )
    
    tool_calls = response['message'].get('tool_calls') or []
    if not tool_calls:
        return response['message'].get('content', "")
    
    messages.append(response['message'])
    
    for tool_call in tool_calls:
        tool_name = tool_call.get('function', {}).get('name', "")
        args = tool_call.get('function', {}).get('arguments', {})
        
        if isinstance(args, str):
            args = json.loads(args)
        
        if tool_name not in TOOLS_WEATHER:
            print(f"Unknown tool: {tool_name}")
            continue
        
        # Get weather data
        weather_result = TOOLS_WEATHER[tool_name]["func"](**args)
        print(f"[Tool: {tool_name}] Executed with args: {args}")
        
        # Automatically convert to markdown
        markdown_result = convert_weather_to_markdown(weather_result)
        
        messages.append({
            "role": "tool",
            "name": tool_name,
            "content": markdown_result
        })
    
    final_response = client.chat(
        model=model_name,
        messages=messages
    )
    
    final_answer = final_response['message'].get('content', "")
    return final_answer


In [34]:

# Test with a weather query
display(Markdown(chat_with_weather_tools_return("What is the weather today in Toronto?")))

[Tool: get_weather] Executed with args: {'city': 'Toronto'}


### 🌡 Current Weather in Toronto

**Today's Conditions:**  
- **Temperature:** -1°C (30°F)  
- **Feels Like:** -7°C (19°F)  
- **Conditions:** Partly cloudy with light snow possible in the evening.  

**Key Updates:**  
- **Morning:** A mix of sun and clouds, with temperatures around 2°C (36°F).  
- **Afternoon:** Cloudy skies, with a high of 1°C (34°F).  
- **Overnight:** Light rain expected, dropping to 3°C (37°F).  

**RealFeel®:**  
- The actual feel is colder due to wind chill, making it feel like -7°C (19°F).  

**Note:** Some sources indicate varying temperatures (e.g., 12°F to 26°F), which may reflect different times or forecasts. For the most accurate current conditions, check the latest updates from local weather services like Environment Canada or AccuWeather. ❄️